In [1]:
# =============================================================================
# GANs — two networks playing a game (forger vs detective)
# =============================================================================
#
# Last notebook (autoencoders / VAE):
#   one model compresses, then redraws. It is graded on "how close is the copy?"
#
# This notebook:
#   two models compete. One tries to FAKE digits/photos. The other tries to
#   CATCH fakes. They get better by fighting each other.
#
#
# ---------------------------------------------------------------------------
# SCENE — art forger and museum detective
# ---------------------------------------------------------------------------
# Forger (Generator, G):
#   Starts with random scribbles (noise). Tries to paint a digit that looks real.
#
# Detective (Discriminator, D):
#   Sees a picture. Says "real" or "fake".
#
# Rules of the game:
#   - Show D real MNIST photos. D should say "real".
#   - Show D G's paintings. D should say "fake".
#   - G only wins if D says "real" on the fakes.
#
# They train in turns. D gets better at catching. G gets better at fooling.
# After enough rounds, G's fakes look like the training photos.
#
#   random noise z  ──G──▶  fake image
#                              │
#                    ┌─────────┴─────────┐
#                    ▼                   ▼
#              D looks at fake      D looks at real MNIST
#              "fake or real?"      "fake or real?"
#
#
# ---------------------------------------------------------------------------
# Why not just use a VAE?
# ---------------------------------------------------------------------------
# VAE: graded on pixel copy + "keep z in town". Sketches often come out
#      blurry (average of many possible digits).
#
# GAN: graded on "does it FOOL the detective?" Sharp details can survive
#      because nobody asked for an average of all 7s — just "a convincing 7".
#
# Cost: training is drama. D too strong → G gives up. G too strong → D
# guesses. You'll see oscillating losses. That's the game, not a bug by itself.
#
#
# ---------------------------------------------------------------------------
# The three names in the title
# ---------------------------------------------------------------------------
#
# 1) DCGAN  (Deep Convolutional GAN)
#    Same forger/detective game, but both nets use conv layers (image brains),
#    not a flat MLP. Better at photos than a pile of Linear layers.
#    Think: "GAN that actually understands grids of pixels."
#
# 2) Conditional GAN  (cGAN)
#    Plain GAN: "draw ANY digit." You don't get to ask for a 3.
#    cGAN: you pass a LABEL (or extra hint) to both G and D.
#      "G, draw a 3."   "D, is this a REAL 3 or a fake 3?"
#    Same game, now with a request slip attached.
#
# 3) Pix2Pix
#    A famous cGAN for "photo in → photo out":
#      edges → handbag photo
#      day street → night street
#      sketch → filled drawing
#    G sees an input image (not only noise). D sees pairs:
#      (input, real target) vs (input, G's output)
#    Think: "conditional GAN where the condition is another picture."
#
#
# ---------------------------------------------------------------------------
# Tiny vocabulary
# ---------------------------------------------------------------------------
#   z / noise     = random numbers G starts from (like "blank canvas")
#   Generator     = forger
#   Discriminator = detective
#   adversarial   = they train AGAINST each other
#   mode collapse = G finds ONE fake that always fools D and repeats it
#                   (a hundred copies of the same 8)
#
#
# ---------------------------------------------------------------------------
# vs GPT / AE (keep them straight)
# ---------------------------------------------------------------------------
#   GPT   finish the next word
#   AE    compress then rebuild the same item
#   VAE   compress to a neighborhood, can invent by sampling z
#   GAN   invent by fooling a detective (no rebuild-the-input homework)

In [ ]:
# =============================================================================
# SETUP — board, dice, and the photo pile before the forger/detective game
# =============================================================================
#
# This cell does not train anyone yet. It only:
#   - imports tools
#   - picks "how big is the blank canvas?" (latent_dim)
#   - loads MNIST photos scaled the way DCGAN likes
#
# Next cells: build Generator + Discriminator, then play the game.
#

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils   # grid of fake images to plot
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)  # same random start → easier to compare reruns
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------------------------------------------------------
# Knobs for the game
# ---------------------------------------------------------------------------
latent_dim = 100   # size of z: how many random numbers the forger starts with
                   # (the "blank canvas". 100 is a DCGAN-paper habit, not magic.)
image_size = 32    # DCGAN stack below wants 32×32 (not raw 28×28)
channels = 1       # gray photos; CIFAR/CelebA would be 3 (RGB)
batch_size = 128   # how many photos D sees in one glance
epochs = 20        # how many times we walk through the whole pile
lr = 0.0002        # small learning rate — GAN games get drunk if this is big
beta1 = 0.5        # Adam's momentum. DCGAN paper uses 0.5 (default is 0.9).
                   # 0.5 = shorter memory, less "keep rushing the same way".

# ---------------------------------------------------------------------------
# Photos: scale pixels to [-1, 1]
# ---------------------------------------------------------------------------
# MNIST ToTensor() gives [0, 1].
# DCGAN generators usually end with Tanh → output in [-1, 1].
# So we shift the REAL photos into the same range, or D would see
# "reals in 0..1" vs "fakes in -1..1" and cheat with a color-range trick.
#
# Normalize([0.5], [0.5])  does  (pixel - 0.5) / 0.5  →  [-1, 1]
#

transform = transforms.Compose(
    [
        transforms.Resize(32),  # 28→32 so G/D conv math lines up (see Generator cell)
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

train_data = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
train_loader = DataLoader(
    train_data, batch_size=batch_size, shuffle=True, num_workers=2
)

print(f"Train samples: {len(train_data)}")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Using device: cpu   (or cuda)
#   Where tensors live. CPU is fine for this tiny MNIST DCGAN; just slower.
#
# Train samples: 60000
#   60k labeled digits. We will IGNORE labels in plain DCGAN (any digit is
#   "real"). Conditional GAN later will use the 0–9 labels as the request slip.
#
# If download fails: check ./data or run with internet once.


Using device: cpu
Train samples: 60000


In [ ]:
# =============================================================================
# GENERATOR — forger: 100 random numbers → a 32×32 "digit"
# =============================================================================
#
# ConvTranspose2d = "upsample with a learned filter" (the opposite of Conv2d).
# We grow a 1×1 blob of z into a full picture:
#
#   z [100] → view as [100, 1, 1]
#         → 4×4  (256 maps)
#         → 8×8
#         → 16×16
#         → 32×32 × 1 channel   Tanh → pixels in [-1, 1]
#
# WHY 32, not 28?
#   These kernel=4, stride=2 layers naturally land on 4,8,16,32.
#   Raw MNIST is 28×28. If D sees 28×28 it shrinks to 3×3, then a 4×4
#   kernel cannot fit → RuntimeError you hit.
#   Fix: Resize(32) on the REAL photos too (and rebuild the DataLoader).
#
# DCGAN habits in this stack:
#   BatchNorm (except last), ReLU, no bias on conv, Tanh at the end.
#

class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1, feature_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_channels = img_channels
        self.feature_dim = feature_dim

        self.main = nn.Sequential(
            # [B, 100, 1, 1] → [B, 256, 4, 4]
            nn.ConvTranspose2d(latent_dim, feature_dim * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.ReLU(True),
            # → [B, 128, 8, 8]
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.ReLU(True),
            # → [B, 64, 16, 16]
            nn.ConvTranspose2d(feature_dim * 2, feature_dim, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(True),
            # → [B, 1, 32, 32]
            nn.ConvTranspose2d(feature_dim, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        z = z.view(z.size(0), self.latent_dim, 1, 1)
        return self.main(z)


# Rebuild the photo pile at 32×32. Changing `transform` alone does NOT
# update an already-built train_loader from the setup cell.
transform = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

_x, _ = next(iter(train_loader))
print(f"real batch shape: {tuple(_x.shape)}  (want [B, 1, 32, 32])")
print(f"fake batch shape: {tuple(Generator()(torch.randn(4, latent_dim)).shape)}")


In [ ]:
# =============================================================================
# DISCRIMINATOR — detective: 32×32 photo → one number "how real?"
# =============================================================================
#
# Conv2d shrinks the picture (mirror of the generator):
#   32×32 → 16×16 → 8×8 → 4×4 → 1×1  then Sigmoid → probability in (0, 1)
#
# LeakyReLU(0.2): tiny negative leak so D doesn't go fully silent.
# No BatchNorm on the first conv (DCGAN paper habit).
# Last layer: 1 channel, 1×1 — "real vs fake" score, not a 10-class digit ID.
#

class Discriminator(nn.Module):
    def __init__(self, img_channels=1, feature_dim=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_dim * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # [B, 1, 32, 32] → [B] probabilities
        return self.main(x).view(-1, 1).squeeze(1)


In [ ]:
# =============================================================================
# WEIGHT INIT — start the game from a small, DCGAN-style shuffle
# =============================================================================
# Random default PyTorch init is OK-ish. DCGAN paper uses:
#   Conv weights  ~ Normal(0, 0.02)   (small)
#   BatchNorm w   ~ Normal(1, 0.02), bias = 0
# Small start → less exploding at the first forger/detective clash.
#

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [ ]:
# =============================================================================
# TRAIN — take turns: teach D, then teach G
# =============================================================================
#
# One batch:
#   1) D sees REAL photos, should say 1.0
#   2) D sees FAKE photos (G's), should say 0.0
#      fake_imgs.detach() → do NOT update G while training D
#   3) G tries to make those same fakes look REAL (label 1.0 to D)
#      now gradients DO flow G ← D
#
# BCELoss = "how wrong was the 0/1 guess?"
# Two Adam opts so each player has their own memory.
#
# fixed_noise: SAME 64 z's every epoch so you can watch fakes improve
# instead of a new random set each time.
#

netG = Generator(latent_dim=latent_dim).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

fixed_noise = torch.randn(64, latent_dim, device=device)
real_label = 1.0
fake_label = 0.0

G_losses, D_losses = [], []

print("Starting Training...")
for epoch in range(epochs):
    for i, (real_imgs, _) in enumerate(train_loader):
        bsz = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # ----- D: real -----
        netD.zero_grad()
        label = torch.full((bsz,), real_label, dtype=torch.float, device=device)
        errD_real = criterion(netD(real_imgs), label)
        errD_real.backward()

        # ----- D: fake -----
        noise = torch.randn(bsz, latent_dim, device=device)
        fake_imgs = netG(noise)
        label.fill_(fake_label)
        errD_fake = criterion(netD(fake_imgs.detach()), label)
        errD_fake.backward()
        errD = errD_real + errD_fake
        optimizerD.step()

        # ----- G: "please call my fakes real" -----
        netG.zero_grad()
        label.fill_(real_label)
        errG = criterion(netD(fake_imgs), label)
        errG.backward()
        optimizerG.step()

        D_losses.append(errD.item())
        G_losses.append(errG.item())

        if i % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] Batch {i}/{len(train_loader)}  "
                f"Loss D: {errD.item():.4f}, Loss G: {errG.item():.4f}"
            )

    with torch.no_grad():
        fake = netG(fixed_noise).detach().cpu()
    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title(f"Fake Images at Epoch {epoch+1}")
    plt.imshow(np.transpose(vutils.make_grid(fake, padding=2, normalize=True), (1, 2, 0)))
    plt.show()

print("Training finished.")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# real batch shape: (128, 1, 32, 32)   ← must be 32, not 28
#
# Loss D / Loss G: they WRESTLE. Not a smooth MiniGPT drop.
#   D ~ 0.5–1.5, G ~ 1–3 is a common messy-but-alive band.
#   D → 0 and G huge: detective unbeatable, forger stuck.
#   D huge, G → 0: forger already fooling; D collapsed.
#
# Epoch grids: epoch 1 = noise/blobs. Later = digit-ish shapes.
# Same layout each epoch because of fixed_noise.


In [ ]:
# =============================================================================
# LOOK BACK — loss traces + a fresh sheet of fakes
# =============================================================================
# G and D lines should bounce, not both crash to 0.
# The grid is NEW random z (not fixed_noise) — extra samples after training.
#

plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

with torch.no_grad():
    noise = torch.randn(64, latent_dim, device=device)
    fake = netG(noise).detach().cpu()

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Generated MNIST Digits")
plt.imshow(np.transpose(vutils.make_grid(fake, padding=2, normalize=True), (1, 2, 0)))
plt.show()

# ---------------------------------------------------------------------------
# HOW TO READ
# ---------------------------------------------------------------------------
# Loss plot: noisy spaghetti is normal for GANs.
# Grid: recognizable digits = win. One repeated blob everywhere = mode collapse.
